## CYCLISTIC CASE STUDY: HOW DOES A BIKE-SHARE NAVIGATE SUCCESS?



### 1. OVERVIEW
#### A. ABOUT THE COMPANY

In 2016, Cyclistic launched a successful bike-share offering. Since then, the program has grown to a fleet of 5,824 bicycles that are geotracked and locked into a network of 692 stations across Chicago. The bikes can be unlocked from one station and returned to any other station in the system anytime.

Until now, Cyclistic's marketing strategy relied on building general awareness and appealing to broad consumer segments. One approach that helped make these things possible was the flexibility of its pricing plans: single-ride passes, full-day passes, and annual memberships. Customers who purchase single-ride or full-day passes are referred to as Casual riders. Customers who purchase annual memberships are Cyclistic Members.

Cyclistic's finance analysts have concluded that annual Members are much more profitable than Casual riders. Although the pricing flexibility helps Cyclistic attract more customers, the director of marketing (Lily Moreno) believes that maximizing the number of annual Members will be key to future growth. Rather than creating a marketing campaign that targets all-new customers, Moreno believes there is a very good chance to convert Casual riders into Members. She notes that Casual riders are already aware of the Cyclistic program and have chosen Cyclistic for their mobility needs. Moreno has set a clear goal: Design marketing strategies aimed at converting Casual riders into annual Members. In order to do that, however, the marketing analyst team needs to better understand how annual Members and Casual riders differ, why Casual riders would buy a membership, and how digital media could affect their marketing tactics. 

#### B. CHARACTERS AND TEAMS

* Cyclistic: A bike-share program that features more than 5,800 bicycles and 600 docking stations. Cyclistic sets itself apart by also offering reclining bikes, hand tricycles, and cargo bikes, making bike-share more inclusive to people with disabilities and riders who can't use a standard two-wheeled bike.
* Lily Moreno: The director of marketing, Moreno is responsible for the development of campaigns and initiatives to promote the bike-share program. These may include email, social media, and other channels.
* Cyclistic marketing analytics team: A team of data analysts who are responsible for collecting, analyzing, and reporting data that helps guide Cyclistic marketing strategy.
* Cyclistic executive team: The notoriously detail-oriented executive team will decide whether to approve the recommended marketing program.

#### C. BUSINESS TASK

Working as a data analyst in the marketing analytics team, I will analyze the Cyclistic historical bike trip data to gain insight into how Casual riders and annual Members use Cyclistic bikes differently. From these insights, the marketing team will design a new strategy to convert Casual riders into annual Members. But first, Cyclistic executives must approve the recommendations, so they must be backed up with compelling data insights and visualizations.


### 2. PREPARE

#### A. DATA SOURCE

The data is public, and it has been made available by Motivate International Inc. under this license [license](https://ride.divvybikes.com/data-license-agreement). The datasets can be downloaded [here](https://divvy-tripdata.s3.amazonaws.com/index.html). There is available data from 2018 but I will use the last twelve months of Cyclistic's historical trip data.; from March 2022 through February 2023.

Note: Although Cyclistic is a fictional company, the datasets are appropriate to answer the business questions.

#### B. ANALYTICAL TOOL

The tool chosen for this analysis is SQL, programmed in BigQuery (Google Cloud). For the visualizations I'll use Tableau.

### Note on BigQuery Access

This notebook demonstrates how to query data using Google BigQuery and Python.

To run this code:
- Replace 'your-project-id' with your Google Cloud project ID
- Replace dataset and table names with your own
- Authenticate using Google Cloud credentials

The dataset used in this project is publicly available (Divvy bike-share data),
but it was loaded into a private BigQuery project for analysis.

This workflow reflects a real-world data pipeline:
**raw data → BigQuery (SQL) → analysis / visualization**

In [ ]:
# =========================================================
# Google BigQuery Setup
# =========================================================

# Identify the project ID in BigQuery
# Replace with your own Google Cloud project ID
PROJECT_ID = 'your-project-id'

# Import the BigQuery Client library
from google.cloud import bigquery

# Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID, location='US')

# =========================================================
# Dataset Reference
# =========================================================

# Replace with your dataset name in BigQuery
DATASET_NAME = 'your_dataset_name'

# Create a reference to the dataset
dataset_ref = client.dataset(DATASET_NAME, project=PROJECT_ID)

# # Fetch dataset metadata
dataset = client.get_dataset(dataset_ref)

# List all tables in the dataset
tables = list(client.list_tables(dataset))

# Print names of all tables in the dataset
for table in tables:  
    print(table.table_id)


* 202203_tripdata.csv
* 202204_tripdata.csv
* 202205_tripdata.csv
* 202206_tripdata.csv
* 202207_tripdata.csv
* 202208_tripdata.csv
* 202209_tripdata.csv
* 202210_tripdata.csv
* 202211_tripdata.csv
* 202212_tripdata.csv
* 202301_tripdata.csv
* 202302_tripdata.csv

The following example demonstrates how to execute a SQL query in BigQuery and retrieve the results using Python.

In [ ]:
# =========================================================
# Example SQL Query
# =========================================================

query = """
SELECT *
FROM 'your-project-id.your_dataset_name.your_table_name'
WHERE start_station_name IS NULL
  AND end_station_name IS NULL
  AND start_station_id IS NULL
  AND end_station_id IS NULL
"""

# Execute query
query_job = client.query(query)

# Convert results to pandas DataFrame
df = query_job.to_dataframe()

# Preview results
df.head()

With the data extracted, we now proceed with the analysis: 
Next, we recognize that these twelve tables have the same schema. Therefore we will use the UNION statement to put them all together in one new table called "combined_tripdata".

In [2]:
import warnings
warnings.filterwarnings('ignore')

query1 = """

CREATE TABLE my-project-010823-374220.cyclistic_data.combined_tripdata AS

  (SELECT * FROM `my-project-010823-374220.cyclistic_data.202203_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202204_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202205_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202206_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202207_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202208_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202209_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202210_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202211_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202212_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202301_tripdata` 
    UNION ALL
  SELECT * FROM `my-project-010823-374220.cyclistic_data.202302_tripdata`); 
   """
# Construct a reference to the "combined_tripdata" table
table_ref = dataset_ref.table("combined_tripdata")

# API request to fetch the table
table = client.get_table(table_ref)

# Preview the first ten lines of the table
client.list_rows(table, max_results=10).to_dataframe()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,20E71EE7F849E218,electric_bike,2022-07-04 08:15:24+00:00,2022-07-04 08:48:19+00:00,None,None,None,None,42.00,-87.71,42.0,-87.70,member
1,9FA1B60F33BE49B4,electric_bike,2022-10-20 16:34:04+00:00,2022-10-20 16:39:43+00:00,None,None,None,None,42.02,-87.67,42.0,-87.66,member
2,710B8497D8DFA76F,electric_bike,2022-10-25 00:39:07+00:00,2022-10-25 01:13:26+00:00,None,None,None,None,42.00,-87.66,42.0,-87.66,casual
3,1199AFD37CF52C2F,electric_bike,2022-09-03 13:17:17+00:00,2022-09-03 13:28:02+00:00,None,None,None,None,42.01,-87.68,42.0,-87.66,casual
4,52D84FC0602FF2C9,electric_bike,2022-10-30 13:42:15+00:00,2022-10-30 13:52:19+00:00,None,None,None,None,41.98,-87.69,42.0,-87.70,casual
5,8C311E758831B6B6,electric_bike,2022-09-17 20:35:56+00:00,2022-09-17 20:57:33+00:00,None,None,None,None,42.03,-87.70,42.0,-87.70,member
6,A1139EBA731B8699,electric_bike,2022-07-03 22:55:12+00:00,2022-07-03 23:09:54+00:00,None,None,None,None,41.97,-87.67,42.0,-87.66,casual
7,C5FC6E0D8260BD87,electric_bike,2022-09-16 05:10:35+00:00,2022-09-16 05:20:11+00:00,None,None,None,None,42.01,-87.67,42.0,-87.68,member
8,D86883BE2B8867C3,electric_bike,2022-08-30 20:57:14+00:00,2022-08-30 21:03:31+00:00,None,None,None,None,42.00,-87.71,42.0,-87.70,casual
9,2F78D9024FC6C1D0,electric_bike,2022-09-10 17:05:13+00:00,2022-09-10 17:40:57+00:00,None,None,None,None,42.00,-87.67,42.0,-87.67,member


In [3]:
table.schema

[SchemaField('ride_id', 'STRING', 'NULLABLE', None, (), None),
 SchemaField('rideable_type', 'STRING', 'NULLABLE', None, (), None),
 SchemaField('started_at', 'TIMESTAMP', 'NULLABLE', None, (), None),
 SchemaField('ended_at', 'TIMESTAMP', 'NULLABLE', None, (), None),
 SchemaField('start_station_name', 'STRING', 'NULLABLE', None, (), None),
 SchemaField('start_station_id', 'STRING', 'NULLABLE', None, (), None),
 SchemaField('end_station_name', 'STRING', 'NULLABLE', None, (), None),
 SchemaField('end_station_id', 'STRING', 'NULLABLE', None, (), None),
 SchemaField('start_lat', 'FLOAT', 'NULLABLE', None, (), None),
 SchemaField('start_lng', 'FLOAT', 'NULLABLE', None, (), None),
 SchemaField('end_lat', 'FLOAT', 'NULLABLE', None, (), None),
 SchemaField('end_lng', 'FLOAT', 'NULLABLE', None, (), None),
 SchemaField('member_casual', 'STRING', 'NULLABLE', None, (), None)]

The "combined_tripdata" table has 5'829,084 rows. It contains all rides that occured during March 2022 through February 2023. Each ride is identified by a unique ride_id, it includes the time when the ride started (started_at), the time when it ended (ended_at), the starting station (start_station_name and start_station_id), the ending station (end_station_name and end_station_id) and the coordinates for both, the start and ending stations (start_lat, start_lng, end_lat, end_lng). The data is grouped by Casual and annual Members (member_casual) and the type of ride (rideable_type) which possible values are electric bike or classic bike.

The fields in the table have the correct data types, except that we will change start_at and ended_at to DATETIME later in the process phase.

### 3. PROCESS

#### A. DATA CLEANING

The following NULL values were found:

* start_station_name has 850,418 NULLS 
* end_station_name has 909,038 NULLS  
* start_station_id 850,550 has NULLS
* end_station_id 909,179 has NULLS 
* end_lat and end_lng have 5,938 NULLS 

We decided to keep all null values in the table because ride id, started_at and ended_at have no nulls. Therefore we assume that these observations are valid trips that need to be counted.

In [4]:
query2="""

SELECT * 
FROM my-project-010823-374220.cyclistic_data.combined_tripdata
WHERE start_station_name IS NULL AND end_station_name IS NULL AND
start_station_id IS NULL AND end_station_id IS NULL;
"""
query_job = client.query(query2)

# API request - run the query, and convert the results to a pandas DataFrame
existing_nulls = query_job.to_dataframe()

# Print the first five rows of the DataFrame
existing_nulls.head(5)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,5A5E78803481ABCF,electric_bike,2022-09-28 10:28:13+00:00,2022-09-28 10:47:24+00:00,None,None,None,None,41.95,-87.83,42.00,-87.83,member
1,FD9202270D0ED76A,electric_bike,2022-08-31 12:46:17+00:00,2022-08-31 13:35:00+00:00,None,None,None,None,41.75,-87.57,41.75,-87.57,casual
2,379525FADB59A9C5,electric_bike,2022-03-21 16:20:05+00:00,2022-03-21 16:48:49+00:00,None,None,None,None,41.74,-87.55,41.75,-87.61,member
3,727C4AE9C08ED981,electric_bike,2022-08-02 08:10:56+00:00,2022-08-02 08:36:53+00:00,None,None,None,None,41.75,-87.57,41.75,-87.57,casual
4,B869AA78E561F9AA,electric_bike,2022-04-16 18:31:19+00:00,2022-04-16 18:45:03+00:00,None,None,None,None,41.76,-87.57,41.75,-87.59,casual


In [5]:
query3="""

SELECT *
FROM my-project-010823-374220.cyclistic_data.combined_tripdata
WHERE end_lat IS NULL AND end_lng IS NULL;
"""
query_job = client.query(query3)

coord_nulls = query_job.to_dataframe()

coord_nulls.head(5)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,A97ACE2D78CDFAB7,classic_bike,2022-11-26 18:37:31+00:00,2022-11-27 19:37:25+00:00,Paulina St & Howard St,515,None,None,42.019159,-87.673573,NaN,NaN,casual
1,7A7511BA12A21290,classic_bike,2022-07-26 10:39:42+00:00,2022-07-27 11:39:35+00:00,Conservatory Dr & Lake St,518,None,None,41.885502,-87.716866,NaN,NaN,casual
2,89B2923C17A9500A,None,2022-10-07 15:07:57+00:00,2022-10-13 13:25:00+00:00,Glenwood Ave & Touhy Ave,525,None,None,42.012701,-87.666058,NaN,NaN,casual
3,D1289A34B8E55A0B,classic_bike,2022-06-27 19:08:54+00:00,2022-06-28 20:08:50+00:00,Laramie Ave & Kinzie St,530,None,None,41.887832,-87.755527,NaN,NaN,casual
4,DD7AD6B8D0A4B002,classic_bike,2022-07-07 19:00:33+00:00,2022-07-08 20:00:27+00:00,Kenton Ave & Madison St,537,None,None,41.880708,-87.741018,NaN,NaN,casual


No duplicates were found when querying ride_ids.

In [6]:
query4="""

SELECT 
  ride_id, COUNT(ride_id) AS rides
FROM my-project-010823-374220.cyclistic_data.combined_tripdata
GROUP BY ride_id
HAVING COUNT(ride_id)>1;
"""
query_job = client.query(query4)

duplicates = query_job.to_dataframe()

duplicates.head()


,ride_id,rides


Inspecting the minimun and maximun values for started_at and ended_at. 

started_at:

* Min= 2022-03-01 00:00:19 UTC
* Max = 2023-02-28 23:59:31 UTC

ended_at:

* Min= 2022-03-01 00:04:30 UTC
* Max = 2023-03-06 15:09:53 UTC

In [7]:
query5="""

SELECT
  MIN(started_at) AS min_started_at,
  MAX(started_at) AS max_started_at
FROM my-project-010823-374220.cyclistic_data.combined_tripdata;
"""
query_job = client.query(query5)

started_dates = query_job.to_dataframe()

started_dates.head()


,min_started_at,max_started_at
0,2022-03-01 00:00:19+00:00,2023-02-28 23:59:31+00:00


In [8]:
query6="""

SELECT
  MIN(ended_at) AS min_ended_at,
  MAX(ended_at) AS max_ended_at
FROM my-project-010823-374220.cyclistic_data.combined_tripdata;
"""
query_job = client.query(query6)

ended_dates = query_job.to_dataframe()

ended_dates.head()


,min_ended_at,max_ended_at
0,2022-03-01 00:04:30+00:00,2023-03-06 15:09:53+00:00


The values found for rideable_type are: Electric_bike, classic_bike and docked_bike.

We assume that all bikes are locked into a station so all bikes are docked bikes. Therefore the value "docked bike" seems to be a mistake, and we're replacing it by NULLs. 

In [9]:
import warnings
warnings.filterwarnings('ignore')

query7="""

SELECT
  rideable_type,
  COUNT (DISTINCT ride_id) AS number_of_trips
FROM my-project-010823-374220.cyclistic_data.combined_tripdata
GROUP BY rideable_type
ORDER BY number_of_trips DESC;

/* UPDATE my-project-010823-374220.cyclistic_data.combined_tripdata 
SET rideable_type=NULL WHERE rideable_type="docked_bike"; */
"""
query_job = client.query(query7)

rideable_count = query_job.to_dataframe()

rideable_count.head()

,rideable_type,number_of_trips
0,electric_bike,2983084
1,classic_bike,2666915
2,None,179085


We create a new table to calculate new fields: Day, weekday_name, month_name, hour_of_day, ride_length_min, dist_meters and dist_miles.

 Also, we're changing started_at and ended_at to DATETIME data type.

In [10]:
import warnings
warnings.filterwarnings('ignore')

query8="""

CREATE TABLE my-project-010823-374220.cyclistic_data.combined_tripdatav2 AS
SELECT 
  ride_id, rideable_type, start_station_name, start_station_id, end_station_name,
  end_station_id, member_casual,start_lat, start_lng, end_lat, end_lng,
  DATE(started_at) AS day,
  FORMAT_DATE('%A', DATE(started_at)) AS weekday_name,
  FORMAT_DATE('%B', DATE(started_at)) AS month_name,
  EXTRACT(HOUR FROM (CAST(started_at AS DATETIME))) AS hour_of_day,
  CAST(ended_at AS DATETIME) AS ended_at_dt,
  CAST(started_at AS DATETIME) AS started_at_dt,
  (DATE_DIFF(CAST(ended_at AS DATETIME), CAST(started_at AS DATETIME), second))/60 AS ride_length_min,
  ST_Distance(ST_GeogPoint(start_lng, start_lat), ST_GeogPoint(end_lng, end_lat)) AS dist_meters,
  (ST_Distance(ST_GeogPoint(start_lng, start_lat), ST_GeogPoint(end_lng, end_lat)))/1609.344 AS dist_miles
FROM my-project-010823-374220.cyclistic_data.combined_tripdata;
"""
table_ref = dataset_ref.table("combined_tripdatav2")

table = client.get_table(table_ref)

client.list_rows(table, max_results=10).to_dataframe()


,ride_id,rideable_type,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,start_lat,start_lng,end_lat,end_lng,day,weekday_name,month_name,hour_of_day,ended_at_dt,started_at_dt,ride_length_min,dist_meters,dist_miles
0,AD200CA94DB59BE3,electric_bike,None,None,None,None,casual,42.01,-87.68,42.0,-87.67,2022-11-11,Friday,November,14,2022-11-11 14:58:11,2022-11-11 14:52:44,5.450000,1385.339883,0.860810
1,44202C98B1DE5B7C,electric_bike,None,None,None,None,casual,41.99,-87.66,42.0,-87.66,2022-07-17,Sunday,July,17,2022-07-17 18:50:05,2022-07-17 17:53:17,56.800000,1111.951012,0.690934
2,D5244A734E83BD15,electric_bike,None,None,None,None,member,42.06,-87.72,42.0,-87.68,2022-07-04,Monday,July,12,2022-07-04 13:02:55,2022-07-04 12:28:23,34.533333,7444.916237,4.626056
3,E1FD7C094F2AFFAD,electric_bike,None,None,None,None,member,42.00,-87.71,42.0,-87.71,2022-07-10,Sunday,July,13,2022-07-10 14:14:51,2022-07-10 13:44:05,30.766667,0.000000,0.000000
4,869FB6F403263FC3,electric_bike,None,None,None,None,member,42.00,-87.66,42.0,-87.66,2022-07-06,Wednesday,July,18,2022-07-06 18:18:38,2022-07-06 18:16:42,1.933333,0.000000,0.000000
5,458C7DC447896739,electric_bike,None,None,None,None,casual,42.01,-87.71,42.0,-87.71,2022-07-23,Saturday,July,22,2022-07-23 22:09:44,2022-07-23 22:03:07,6.616667,1111.951012,0.690934
6,E96A2C57C85FCFD6,electric_bike,None,None,None,None,casual,41.98,-87.71,42.0,-87.69,2022-04-24,Sunday,April,15,2022-04-24 15:43:52,2022-04-24 15:33:29,10.383333,2770.912119,1.721765
7,D8DF16E0DF0B9B44,electric_bike,None,None,None,None,member,41.99,-87.67,42.0,-87.70,2022-03-21,Monday,March,0,2022-03-21 00:15:46,2022-03-21 00:03:16,12.500000,2717.158514,1.688364
8,9CD15AF3622BA19F,electric_bike,None,None,Public Rack - Western Ave & Devon Ave,931,member,42.00,-87.66,42.0,-87.69,2022-06-10,Friday,June,22,2022-06-10 22:28:38,2022-06-10 22:19:23,9.250000,2479.021909,1.540393
9,3D2C77D878453566,electric_bike,None,None,None,None,member,41.98,-87.66,42.0,-87.67,2022-09-23,Friday,September,13,2022-09-23 13:22:41,2022-09-23 13:13:47,8.900000,2372.507883,1.474208


We found a problem with ride_length (calculated in minutes). The maximun value is 29 days and the minimun is a negative number.

In [11]:
query9="""

SELECT
  AVG(ride_length_min) AS mean_ride_length,
  MAX(ride_length_min) AS max_ride_length,
  MIN(ride_length_min) AS min_ride_length
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav2;
"""
query_job = client.query(query9)

rideable_count = query_job.to_dataframe()

rideable_count.head()

,mean_ride_length,max_ride_length,min_ride_length
0,19.217434,41387.25,-10353.35


We filter any ride length that is negative, and over 7 days. We assume that after 7 days the bike is considered lost. 

We filter 1,054 observations.

In [12]:
query10="""
CREATE TABLE my-project-010823-374220.cyclistic_data.combined_tripdatav3 AS
SELECT *
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav2
WHERE ride_length_min >0 AND ride_length_min <= 10080;
"""
table_ref = dataset_ref.table("combined_tripdatav3")

table = client.get_table(table_ref)

client.list_rows(table, max_results=10).to_dataframe()

,ride_id,rideable_type,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,start_lat,start_lng,end_lat,end_lng,day,weekday_name,month_name,hour_of_day,ended_at_dt,started_at_dt,ride_length_min,dist_meters,dist_miles
0,C3C9652F336344D3,electric_bike,None,None,None,None,member,42.00,-87.67,42.0,-87.67,2022-09-06,Tuesday,September,20,2022-09-06 20:38:23,2022-09-06 20:26:11,12.200000,0.000000,0.000000
1,AAB2B18C19B9E99C,electric_bike,None,None,None,None,casual,42.00,-87.67,42.0,-87.67,2022-08-27,Saturday,August,19,2022-08-27 19:10:05,2022-08-27 19:09:54,0.183333,0.000000,0.000000
2,2D6DA95AA736089A,electric_bike,None,None,None,None,casual,41.99,-87.67,42.0,-87.67,2022-12-11,Sunday,December,16,2022-12-11 16:50:06,2022-12-11 16:46:08,3.966667,1111.951012,0.690934
3,8E67A84F7428B0F1,electric_bike,None,None,None,None,casual,42.00,-87.70,42.0,-87.70,2022-05-13,Friday,May,18,2022-05-13 19:01:18,2022-05-13 18:56:46,4.533333,0.000000,0.000000
4,9B72F56AAACF0971,electric_bike,None,None,None,None,casual,41.95,-87.66,42.0,-87.86,2022-07-25,Monday,July,20,2022-07-25 22:01:41,2022-07-25 20:47:47,73.900000,17443.074299,10.838624
5,4D00CB831F98048F,electric_bike,None,None,None,None,member,42.00,-87.70,42.0,-87.69,2022-05-13,Friday,May,19,2022-05-13 20:01:14,2022-05-13 19:57:53,3.350000,826.340640,0.513464
6,1D9F7400DFD86FDA,electric_bike,None,None,None,None,casual,42.00,-87.70,42.0,-87.70,2022-06-13,Monday,June,15,2022-06-13 15:35:36,2022-06-13 15:26:50,8.766667,0.000000,0.000000
7,92240F641A125443,electric_bike,None,None,None,None,casual,42.01,-87.67,42.0,-87.67,2022-05-29,Sunday,May,14,2022-05-29 14:50:56,2022-05-29 14:45:42,5.233333,1111.951012,0.690934
8,F9B2B0E398E6A8C6,electric_bike,None,None,None,None,casual,42.00,-87.70,42.0,-87.70,2022-07-19,Tuesday,July,20,2022-07-19 20:13:20,2022-07-19 20:13:07,0.216667,0.000000,0.000000
9,A681B06CAB8A4EC1,electric_bike,None,None,None,None,member,42.00,-87.71,42.0,-87.70,2022-07-07,Thursday,July,23,2022-07-07 23:49:50,2022-07-07 23:42:02,7.800000,826.340640,0.513464


It's suggested to check if 0.017 minutes is a valid value. For now, we assume it is correct.

In [13]:
query11="""

SELECT
  AVG(ride_length_min) AS mean_ride_length,
  MAX(ride_length_min) AS max_ride_length,
  MIN(ride_length_min) AS min_ride_length
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3;
"""
query_job = client.query(query11)

rideable_count = query_job.to_dataframe()

rideable_count.head()

,mean_ride_length,max_ride_length,min_ride_length
0,17.84041,10040.383333,0.016667


### 4. ANALYZE

The query below pulls data by Member and Casual rides. We look at number of trips, average ride length and average distance per group. Length is measured in minutes and distance in miles.

The calculated field distance in miles is not accurate. The distance is calculated as the difference between the coordinates of the starting point and the ending point of a trip without considering for example, the extra miles riders take when going around town.

In [14]:
import warnings
warnings.filterwarnings('ignore')

query12="""

SELECT 
    member_casual,
    COUNT (DISTINCT ride_id) AS number_of_trips,
    AVG(ride_length_min) AS mean_ride_length_min,
    AVG(dist_miles) AS mean_dist_miles
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3
GROUP BY member_casual;
"""
query_job = client.query(query12)

rideable_count = query_job.to_dataframe()

rideable_count.head()

,member_casual,number_of_trips,mean_ride_length_min,mean_dist_miles
0,member,3463689,12.581284,1.304863
1,casual,2364341,25.544872,1.347325


The query below is grouping the data by member_casual and rideable_type. It shows the number of trips and average ride length by bike type (electric and classic).

In [15]:
import warnings
warnings.filterwarnings('ignore')

query13="""

SELECT 
    member_casual,rideable_type,
    COUNT (DISTINCT ride_id) AS number_of_trips,
    AVG(ride_length_min) AS mean_ride_length_min
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3
GROUP BY member_casual, rideable_type
HAVING rideable_type IS NOT NULL
ORDER BY number_of_trips DESC;
"""
query_job = client.query(query13)

rideable_count = query_job.to_dataframe()

rideable_count.head()

,member_casual,rideable_type,number_of_trips,mean_ride_length_min
0,member,classic_bike,1761021,13.765388
1,member,electric_bike,1702668,11.356599
2,casual,electric_bike,1280009,15.997734
3,casual,classic_bike,905769,28.561918


The query below pulls data by month for both Member and Casual rides. We're looking into number of trips and average ride length in minutes.

In [16]:
import warnings
warnings.filterwarnings('ignore')

query14="""

SELECT 
    member_casual,month_name,
    COUNT (DISTINCT ride_id) AS number_of_trips,
    AVG(ride_length_min) AS mean_ride_length_min
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3
GROUP BY member_casual,month_name
ORDER BY member_casual, number_of_trips DESC;
"""
query_job = client.query(query14)

rideable_count = query_job.to_dataframe()

rideable_count.head(24)

,member_casual,month_name,number_of_trips,mean_ride_length_min
0,casual,July,405948,26.699072
1,casual,June,368914,27.476252
2,casual,August,358800,25.613256
3,casual,September,296583,23.868264
4,casual,May,280352,29.009943
5,casual,October,208909,21.706275
6,casual,April,126380,27.601541
7,casual,November,100726,19.126215
8,casual,March,89854,29.736294
9,casual,December,44879,18.318618


The query below pulls data by day of the week for both Member and Casual rides. We're looking into number of trips and average ride length in minutes.

In [17]:
query15="""

SELECT 
    member_casual,weekday_name,
    COUNT (DISTINCT ride_id) AS number_of_trips,
    AVG(ride_length_min) AS mean_ride_length_min
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3
GROUP BY member_casual,weekday_name
ORDER BY member_casual, number_of_trips DESC;
"""
query_job = client.query(query15)

rideable_count = query_job.to_dataframe()

rideable_count.head(14)

,member_casual,weekday_name,number_of_trips,mean_ride_length_min
0,casual,Saturday,478296,28.299942
1,casual,Sunday,398464,29.184926
2,casual,Friday,338703,24.540990
3,casual,Thursday,313563,23.019658
4,casual,Monday,283240,25.809558
5,casual,Wednesday,279839,22.001323
6,casual,Tuesday,272236,22.901267
7,member,Thursday,547354,12.156972
8,member,Tuesday,546578,11.935146
9,member,Wednesday,542401,11.971836


The query below pulls data by hour of the day for both Member and Casual rides. We're looking into number of trips and average ride length in minutes.

In [18]:
query16="""

SELECT 
    member_casual,hour_of_day,
    COUNT (DISTINCT ride_id) AS number_of_trips,
    AVG(ride_length_min) AS mean_ride_length_min
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3
GROUP BY member_casual,hour_of_day
ORDER BY member_casual, hour_of_day;
"""
query_job = client.query(query16)

rideable_count = query_job.to_dataframe()

rideable_count.head(48)


,member_casual,hour_of_day,number_of_trips,mean_ride_length_min
0,casual,0,46938,24.718709
1,casual,1,30363,29.497129
2,casual,2,18837,29.266814
3,casual,3,11174,31.767287
4,casual,4,7747,28.547571
5,casual,5,12792,19.655894
6,casual,6,30566,17.495425
7,casual,7,53361,16.759088
8,casual,8,71950,18.739343
9,casual,9,73913,23.720995


The query below is extracting the top 200 routes for Members and Casual rides. Routes were taken as unique combinations of start and end stations grouped by the number of times that route was ridden during the year. The trips taken in the top 200 routes accounted for about 7% of all total trips for Casual rides, and 5% for Member rides. 

We excluded 5,938 NULLS found for end_lat and end_lng.

In [19]:
query17="""

SELECT 
   member_casual, TRIM(start_station_name) AS new_start_station_name, 
   TRIM(start_station_id) AS new_start_station_id,
   TRIM(end_station_name) AS new_end_station_name,
   COUNT(ride_id) AS number_of_trips,
   start_lat, start_lng, end_lat, end_lng
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3
GROUP BY member_casual, new_start_station_name, start_station_id, new_end_station_name, start_lat, start_lng, end_lat, end_lng
HAVING member_casual="casual" AND new_start_station_name IS NOT NULL AND end_lat IS NOT NULL AND
   end_lng IS NOT NULL
ORDER BY number_of_trips DESC
LIMIT 500;
"""
query_job = client.query(query17)

rideable_count = query_job.to_dataframe()

rideable_count.head(10)

,member_casual,new_start_station_name,new_start_station_id,new_end_station_name,number_of_trips,start_lat,start_lng,end_lat,end_lng
0,casual,Streeter Dr & Grand Ave,13022,Streeter Dr & Grand Ave,8094,41.892278,-87.612043,41.892278,-87.612043
1,casual,DuSable Lake Shore Dr & Monroe St,13300,DuSable Lake Shore Dr & Monroe St,4883,41.880958,-87.616743,41.880958,-87.616743
2,casual,DuSable Lake Shore Dr & Monroe St,13300,Streeter Dr & Grand Ave,3946,41.880958,-87.616743,41.892278,-87.612043
3,casual,Michigan Ave & Oak St,13042,Michigan Ave & Oak St,2321,41.900960,-87.623777,41.900960,-87.623777
4,casual,Streeter Dr & Grand Ave,13022,DuSable Lake Shore Dr & Monroe St,2208,41.892278,-87.612043,41.880958,-87.616743
5,casual,Montrose Harbor,TA1308000012,Montrose Harbor,2117,41.963982,-87.638181,41.963982,-87.638181
6,casual,DuSable Lake Shore Dr & North Blvd,LF-005,DuSable Lake Shore Dr & North Blvd,1871,41.911722,-87.626804,41.911722,-87.626804
7,casual,Millennium Park,13008,Millennium Park,1814,41.881032,-87.624084,41.881032,-87.624084
8,casual,Theater on the Lake,TA1308000001,Theater on the Lake,1768,41.926277,-87.630834,41.926277,-87.630834
9,casual,Streeter Dr & Grand Ave,13022,DuSable Lake Shore Dr & North Blvd,1751,41.892278,-87.612043,41.911722,-87.626804


In [20]:
query18="""

SELECT 
   member_casual, TRIM(start_station_name) AS new_start_station_name, 
   TRIM(start_station_id) AS new_start_station_id,
   TRIM(end_station_name) AS new_end_station_name,
   COUNT(ride_id) AS number_of_trips,
   start_lat, start_lng, end_lat, end_lng
FROM my-project-010823-374220.cyclistic_data.combined_tripdatav3
GROUP BY member_casual, new_start_station_name, start_station_id, new_end_station_name, start_lat, start_lng, end_lat, end_lng
HAVING member_casual="member" AND new_start_station_name IS NOT NULL AND end_lat IS NOT NULL AND
   end_lng IS NOT NULL
ORDER BY number_of_trips DESC
LIMIT 500;
"""
query_job = client.query(query18)

rideable_count = query_job.to_dataframe()

rideable_count.head(10)

,member_casual,new_start_station_name,new_start_station_id,new_end_station_name,number_of_trips,start_lat,start_lng,end_lat,end_lng
0,member,Ellis Ave & 60th St,KA1503000014,University Ave & 57th St,5546,41.785097,-87.601073,41.791478,-87.599861
1,member,University Ave & 57th St,KA1503000071,Ellis Ave & 60th St,5202,41.791478,-87.599861,41.785097,-87.601073
2,member,Ellis Ave & 60th St,KA1503000014,Ellis Ave & 55th St,4799,41.785097,-87.601073,41.794301,-87.601450
3,member,Ellis Ave & 55th St,KA1504000076,Ellis Ave & 60th St,4518,41.794301,-87.601450,41.785097,-87.601073
4,member,State St & 33rd St,13216,Calumet Ave & 33rd St,2415,41.834734,-87.625813,41.834900,-87.617930
5,member,Calumet Ave & 33rd St,13217,State St & 33rd St,2312,41.834900,-87.617930,41.834734,-87.625813
6,member,University Ave & 57th St,KA1503000071,Kimbark Ave & 53rd St,2006,41.791478,-87.599861,41.799568,-87.594747
7,member,Ellis Ave & 60th St,KA1503000014,Ellis Ave & 58th St,1881,41.785097,-87.601073,41.788746,-87.601334
8,member,Kimbark Ave & 53rd St,TA1309000037,University Ave & 57th St,1796,41.799568,-87.594747,41.791478,-87.599861
9,member,Ellis Ave & 58th St,TA1309000011,Ellis Ave & 60th St,1656,41.788746,-87.601334,41.785097,-87.601073


### 6. ACT

#### A. CONCLUSIONS

As a reminder, customers who purchase single-ride or full-day passes are referred to as Casual riders. Customers who purchase annual memberships are Cyclistic Members.

- 59.3% of total rides were Member trips, and 40.6% were Casual rides.

- The average ride length in minutes was 12.58 for Members and 25.54 for Casuals. This seems to indicate that Members usually ride bikes to commute while Casuals ride mostly for tourism and leisure, taking more time per ride.

- The calculated field distance in miles is not accurate. The distance is calculated as the difference between the coordinates of the starting point and the ending point of a trip. Casuals might start in one station and go all around town before ending their trip at an ending station. The distance for Casuals is the distance between starting and ending points without considering for example, the extra miles they took when going around town.

- Casual rides showed a preference for electric bikes: 58.6% of all rides used electric bikes while 41.4% used classic bikes. Casual riders seem to prefer electric bikes because these are faster, more reliable for longer trips and easier to use. Members on the other hand used both electric and classic bikes almost at the same rate, 50.8% of rides used electric bikes while 49% used classic bikes.

- For Casuals, the number of rides peak during the summer months of June, July and August. Casual rides seem to be closely related to the weather, there is also activity between the spring months of April and May, and the fall months of September and October. During the colder months of winter (December, January and February) Casual rides declined sharply to less than 50,000 rides a month.

- Member rides tend to be more consistent throughout the year with the highest months being July and August. There is a similar pattern to that of Casual rides, however the colder months in winter show much more activity with about 140,000 rides per month.

- In average time per ride, Casual rides peak in the month of March with 30 minutes per ride. The months of March through August have over 25 minutes of average ride duration. For Member rides the average length is more consistent throughout the year fluctuating between 10 and 14 minutes. This pattern seems closer to a commuter’s time slightly peaking during the summer months.

- Casual rides peak on Saturdays and Sundays at over 400,000 rides. During the week from Monday through Friday the number of rides stay at around 300,000 rides. For Members the opposite happens, Saturday and Sunday are the days with the lowest number of rides, while from Monday through Friday this number stays around 500,000 rides.

- Average length per ride follows the same pattern as number of trips. During the weekend casuals take longer trips between 28 and 29 minutes. Member rides are more consistent at between 12 to 14 minutes during the entire week.

- Most rides for both groups (Members and Casuals) take place in the afternoon between the hours of 4pm and 6pm. Members follow a working schedule with most trips happening during the commuting time of 4pm to 6pm. Although the morning hours have less activity, the commuting hours between 6am and 8am have about three times as many trips as Casual rides.

- For average time per ride, Member rides are more consistent with 10 to 13 minutes throughout the day.  For Casuals, late morning and early afternoon rides (between 10am and 3pm) tend to be longer at 29 to 27 minutes.  For Casuals, the average length ride peaks at 32 minutes at 3am. This is due to the presence of 7 outliers that report trips for more than 5,000 hours at the hour of 3am. We would need to investigate that to validate this data. 

- The top 200 routes for rides, were taken as unique combinations of start and end stations grouped by the number of times that route was ridden during the year. The trips taken in the top 200 routes accounted for about 7% of all total trips for Casual rides, and 5% for Member rides. 

- Casual routes are concentrated in the downtown area of Chicago, with the station at Streeter Dr. and Grand Ave. being the start station that originated the most routes. This shows that most Casual rides are taken by tourists and visitors looking to get around downtown to visit local attractions.

- On the other hand, Member trips were taken across a wider section of the map, reaching farther areas from the center of the city. In fact, the top Member route was taken from the start station of University Ave and 57th St. which is located at Hyde Park. This makes sense because most of Members are commuters.

#### B. RECOMMENDATIONS

We concluded that Members and Casual riders are two different customer segments that have different needs. Converting some Casual riders into Members might be possible if there is a group of Casual riders that use the bikes to commute but don’t want to commit to an annual membership. It would be recommendable to have a marketing strategy that communicates to these Casual riders that they can save money with a membership. The next step though would be to have a customer survey to find out how often this is happening.

However, the group from Casual riders that acquire a membership will probably not be a massive move. We could estimate how big this group is by the results of the customer survey and then analyze financial information such as cost and profits. This way we could better measure the financial impact of converting some Casual riders to annual Members. 

We believe that keeping pricing flexibility is very important. Communicating the benefits of each pricing plan is critical as well. Casual riders can be engaged by digital marketing through social media. Also, campaigns such as Google Display would be useful to advertise in thousands of websites with touristic content. 